# Demand Forecasting — Step 1: Data Understanding

This notebook inspects and validates the two Gold tables in Unity Catalog (`oil_gas_demo.gold.daily_demand_agg` and `oil_gas_demo.gold.demand_features`) **before any model training**. We verify the runtime environment, table schemas, row counts, date ranges, null values, duplicate keys, date continuity per product-city, lag/rolling semantics, and potential target leakage — all prerequisites for building a trustworthy 7-day-ahead demand forecasting model.

**Workflow overview (later steps will be added below):**
1. **Step 1** — Data Understanding *(this notebook section)*
2. Step 2 — Target creation (7-calendar-day-ahead join)
3. Step 3 — Feature Store table
4. Step 4 — Chronological time split (64% train / 16% val / 20% test)
5. Step 5 — Random Forest training with scikit-learn Pipeline
6. Step 6 — MLflow experiment tracking
7. Step 7 — Evaluation (MAE, RMSE, baseline comparison)
8. Step 8 — Unity Catalog Model Registry

**Rule:** Do not overwrite or delete any existing Bronze, Silver, or Gold tables.

In [0]:
# ============================================================
# CELL 1: Runtime, Libraries & Environment Check
# ============================================================
# Before touching any data, we confirm what compute, Python,
# Spark, and ML libraries are available.
# This prevents surprises later (e.g., missing scikit-learn
# or an incompatible Feature Store API).
#
# WHAT TO VERIFY after running:
#   • Spark version >= 3.4 (needed for serverless / Spark Connect)
#   • scikit-learn is installed (version >= 1.0)
#   • mlflow is installed (version >= 2.0 for UC model registry)
#   • databricks.feature_engineering imports successfully
#   • If any library shows ✗, install it with:
#       %pip install <library-name>
#       dbutils.library.restartPython()
# ------------------------------------------------------------

import sys

print("=" * 60)
print("RUNTIME & ENVIRONMENT CHECK")
print("=" * 60)

# --- Spark version (confirms we have a running SparkSession) ---
print(f"\nSpark version : {spark.version}")

# --- Python version ---
print(f"Python version: {sys.version.split()[0]}")

# --- Check key libraries one by one ---
# We wrap each import in try/except so that one missing library
# does not crash the whole cell.
libraries_to_check = {
    "pandas":                "pandas",
    "numpy":                 "numpy",
    "scikit-learn":          "sklearn",
    "mlflow":                "mlflow",
    "feature-engineering":  "databricks.feature_engineering",
}

print("\n--- Library Availability ---")
for display_name, import_name in libraries_to_check.items():
    try:
        module = __import__(import_name, fromlist=["__version__"])
        version = getattr(module, "__version__", "version not available")
        print(f"  ✓ {display_name:<22} → {version}")
    except ImportError:
        print(f"  ✗ {display_name:<22} → NOT INSTALLED")

print("\n If any library shows ✗, install it in a new cell:")
print("   %pip install <library-name>")
print("   Then run: dbutils.library.restartPython()")

In [0]:
# ============================================================
# CELL 2: Load Both Gold Tables & Print Schemas
# ============================================================
# We read both Unity Catalog tables into Spark DataFrames and
# print their schemas (column names + data types).
# This confirms:
#   1. The tables exist and we have SELECT permission.
#   2. The columns match what we expect.
#   3. Data types are correct (e.g., transaction_date is DATE).
#
# WHAT TO VERIFY after running:
#   • daily_demand_agg has: transaction_date, product_name,
#     destination_city, total_demand, avg_unit_price,
#     total_inventory, transaction_count
#   • demand_features has all of the above PLUS: day_of_week,
#     month, demand_lag_1, demand_lag_7, rolling_avg_7,
#     rolling_std_7
#   • transaction_date is a DATE or TIMESTAMP (not a string)
#   • Numeric columns are DOUBLE, INT, or BIGINT (not strings)
#
# COMMON ERROR: "Table does not exist" or "UNAUTHORIZED"
#   → You may lack USE CATALOG or SELECT permission.
#   → Contact your workspace admin to grant:
#     GRANT USE CATALOG ON CATALOG oil_gas_demo TO <user>
#     GRANT USE SCHEMA ON SCHEMA oil_gas_demo.gold TO <user>
#     GRANT SELECT ON TABLE oil_gas_demo.gold.daily_demand_agg TO <user>
# ------------------------------------------------------------

print("=" * 60)
print("LOADING GOLD TABLES")
print("=" * 60)

# --- Load the aggregation table (actual daily demand) ---
# This table has the raw daily totals per product and city.
try:
    agg_df = spark.read.table("oil_gas_demo.gold.daily_demand_agg")
    print("\n✓ Loaded: oil_gas_demo.gold.daily_demand_agg")
except Exception as e:
    print(f"\n✗ ERROR loading daily_demand_agg: {e}")
    raise

# --- Load the feature table (features built on top of demand) ---
# This table adds day_of_week, month, lags, and rolling stats.
try:
    feat_df = spark.read.table("oil_gas_demo.gold.demand_features")
    print("✓ Loaded: oil_gas_demo.gold.demand_features")
except Exception as e:
    print(f"✗ ERROR loading demand_features: {e}")
    raise

# --- Print schemas so we can verify column names and types ---
print("\n" + "=" * 60)
print("SCHEMA: daily_demand_agg")
print("=" * 60)
agg_df.printSchema()

print("\n" + "=" * 60)
print("SCHEMA: demand_features")
print("=" * 60)
feat_df.printSchema()

# --- Show a few sample rows from each table ---
print("\n--- Sample rows from daily_demand_agg (first 5) ---")
display(agg_df.limit(5))

print("\n--- Sample rows from demand_features (first 5) ---")
display(feat_df.limit(5))

In [0]:
# ============================================================
# CELL 3: Row Counts, Date Ranges & Missing Values
# ============================================================
# We get the big-picture stats for both tables:
#   - Total row count (tells us dataset size)
#   - Min and max transaction_date (tells us time span)
#   - Distinct products and cities (tells us how many time series)
#   - Null count per column (tells us data quality)
#
# We build all null-count expressions in a SINGLE .agg() call
# so Spark scans the table once instead of once-per-column.
# This is a best practice for large tables.
#
# WHAT TO VERIFY after running:
#   • Enough date history: ideally 3+ months per combo for a
#     meaningful 7-day-ahead forecast. More is better.
#   • Lag/rolling columns WILL have nulls for the first few rows
#     of each series — that is EXPECTED.
#   • total_demand, avg_unit_price, total_inventory should have
#     ZERO nulls. If they have nulls, investigate upstream.
#   • Note the total row count. Under ~500K → safe for Pandas
#     conversion later (Step 5). Millions → we'll need a
#     different approach (e.g., Spark ML or sampling).
# ------------------------------------------------------------

from pyspark.sql import functions as F

print("=" * 60)
print("OVERVIEW: daily_demand_agg")
print("=" * 60)

# --- Single-pass summary for agg_df ---
agg_summary = agg_df.agg(
    F.count("*").alias("row_count"),
    F.min("transaction_date").alias("min_date"),
    F.max("transaction_date").alias("max_date"),
    F.countDistinct("product_name").alias("n_products"),
    F.countDistinct("destination_city").alias("n_cities"),
    F.countDistinct("product_name", "destination_city").alias("n_combos"),
).collect()[0]

print(f"  Total rows           : {agg_summary['row_count']:,}")
print(f"  Date range           : {agg_summary['min_date']}  →  {agg_summary['max_date']}")
print(f"  Distinct products    : {agg_summary['n_products']}")
print(f"  Distinct cities      : {agg_summary['n_cities']}")
print(f"  Product-city combos  : {agg_summary['n_combos']}")

# --- Null counts for agg_df (single pass) ---
print("\n  --- Null counts (daily_demand_agg) ---")
null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in agg_df.columns]
agg_nulls = agg_df.agg(*null_exprs).collect()[0].asDict()
for c in agg_df.columns:
    print(f"    {c:<25} : {agg_nulls[c]}")

print("\n" + "=" * 60)
print("OVERVIEW: demand_features")
print("=" * 60)

# --- Single-pass summary for feat_df ---
feat_summary = feat_df.agg(
    F.count("*").alias("row_count"),
    F.min("transaction_date").alias("min_date"),
    F.max("transaction_date").alias("max_date"),
    F.countDistinct("product_name").alias("n_products"),
    F.countDistinct("destination_city").alias("n_cities"),
    F.countDistinct("product_name", "destination_city").alias("n_combos"),
).collect()[0]

print(f"  Total rows           : {feat_summary['row_count']:,}")
print(f"  Date range           : {feat_summary['min_date']}  →  {feat_summary['max_date']}")
print(f"  Distinct products    : {feat_summary['n_products']}")
print(f"  Distinct cities      : {feat_summary['n_cities']}")
print(f"  Product-city combos  : {feat_summary['n_combos']}")

# --- Null counts for feat_df (single pass) ---
print("\n  --- Null counts (demand_features) ---")
null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in feat_df.columns]
feat_nulls = feat_df.agg(*null_exprs).collect()[0].asDict()
for c in feat_df.columns:
    flag = " ⚠️" if feat_nulls[c] > 0 and c not in (
        "demand_lag_1", "demand_lag_7", "rolling_avg_7", "rolling_std_7"
    ) else ""
    print(f"    {c:<25} : {feat_nulls[c]}{flag}")

print("""
📌 NOTES:
  • Nulls in lag_1, lag_7, rolling_avg_7, rolling_std_7 are EXPECTED
    for the first few rows of each time series (they need history).
  • Nulls in total_demand, avg_unit_price, or total_inventory are
    a PROBLEM — investigate the upstream Gold pipeline.
  • If row_count is under ~500K, Pandas conversion in Step 5 is safe.
""")

In [0]:
# ============================================================
# CELL 4: Duplicate Key Check
# ============================================================
# The composite key (transaction_date, product_name,
# destination_city) must be unique in both tables.
# Duplicate keys would cause:
#   - Ambiguous joins in Step 2 (target creation)
#   - Extra rows silently corrupting training data
#
# WHAT TO VERIFY after running:
#   • Both tables should show ZERO duplicate rows.
#   • If duplicates exist, you must resolve them before
#     proceeding (e.g., keep the latest, or aggregate them).
# ------------------------------------------------------------

print("=" * 60)
print("DUPLICATE KEY CHECK")
print("=" * 60)

key_cols = ["transaction_date", "product_name", "destination_city"]

for name, df in [("daily_demand_agg", agg_df), ("demand_features", feat_df)]:
    print(f"\n--- {name} ---")

    # Group by the composite key and count rows per group
    dup_counts = (df
        .groupBy(key_cols)
        .agg(F.count("*").alias("row_count"))
        .filter(F.col("row_count") > 1)
        .orderBy(F.desc("row_count")))

    dup_rows = dup_counts.collect()

    if len(dup_rows) == 0:
        print(f"  ✅ No duplicate keys found. ({name} is clean.)")
    else:
        print(f"  ⚠️ Found {len(dup_rows)} duplicate key combinations!")
        print("  First 10 duplicates:")
        for row in dup_rows[:10]:
            print(f"    {row['transaction_date']} | {row['product_name']} | {row['destination_city']} → {row['row_count']} rows")
        print("\n  ⚠️ ACTION NEEDED: Resolve duplicates before proceeding.")
        print("  Options: keep latest, average, or re-aggregate upstream.")

In [0]:
# ============================================================
# CELL 5: Date Continuity Per Product-City Combination
# ============================================================
# For each product-city time series, we check whether dates are
# continuous (every calendar day is present) or have gaps.
#
# WHY THIS MATTERS:
#   • demand_lag_7 should mean "demand 7 CALENDAR days ago",
#     not "demand 7 rows ago." If dates are missing, a 7-row
#     shift ≠ 7 calendar days.
#   • Missing dates also affect the 7-day-ahead target join
#     in Step 2 — if the target date doesn't exist, we can't
#     create a training row for that example.
#
# APPROACH:
#   1. For each combo, compute expected days = max_date - min_date + 1
#   2. Compare expected days vs. actual row count
#   3. If they differ, there are missing calendar dates
#   4. Also check consecutive-row date gaps with a window function
#
# WHAT TO VERIFY after running:
#   • Each combo should have row_count == expected_days
#   • If any combo has fewer rows than expected, there are gaps
#   • Note which combos have gaps — this affects Step 2 and Step 5
# ------------------------------------------------------------

from pyspark.sql.window import Window

print("=" * 60)
print("DATE CONTINUITY CHECK")
print("=" * 60)

# --- Get all distinct product-city combinations ---
combos = (feat_df
    .select("product_name", "destination_city")
    .distinct()
    .orderBy("product_name", "destination_city")
    .collect())

print(f"\nTotal product-city combinations: {len(combos)}")
print("\nChecking first 5 combinations (and flagging all with gaps):\n")

combos_with_gaps = []

for i, combo in enumerate(combos):
    prod = combo["product_name"]
    city = combo["destination_city"]

    # Filter to this combo and get summary stats
    subset = feat_df.filter(
        (F.col("product_name") == prod) &
        (F.col("destination_city") == city)
    )

    stats = subset.agg(
        F.count("*").alias("n_rows"),
        F.min("transaction_date").alias("min_d"),
        F.max("transaction_date").alias("max_d"),
    ).collect()[0]

    n_rows = stats["n_rows"]
    min_d = stats["min_d"]
    max_d = stats["max_d"]
    expected_days = (max_d - min_d).days + 1
    missing_days = expected_days - n_rows

    # Print details for first 5 combos
    if i < 5:
        print(f"  {prod} / {city}:")
        print(f"    Rows: {n_rows}  |  Dates: {min_d} → {max_d}  |  Expected: {expected_days} days")
        if missing_days == 0:
            print(f"    ✅ Dates are continuous (no gaps)")
        else:
            print(f"    ⚠️ {missing_days} missing calendar days!")

        # Check specific date gaps using a window function
        w = Window.orderBy("transaction_date")
        gaps_df = (subset
            .withColumn("prev_date", F.lag("transaction_date").over(w))
            .withColumn("gap_days", F.datediff(F.col("transaction_date"), F.col("prev_date")))
            .filter(F.col("gap_days") > 1)
            .select("prev_date", "transaction_date", "gap_days"))

        gap_rows = gaps_df.collect()
        if gap_rows:
            print(f"    Gap details ({len(gap_rows)} gaps):")
            for g in gap_rows[:5]:
                print(f"      {g['prev_date']} → {g['transaction_date']}  ({g['gap_days']} days)")
            if len(gap_rows) > 5:
                print(f"      ... and {len(gap_rows) - 5} more gaps")
        print()

    # Track combos with gaps
    if missing_days > 0:
        combos_with_gaps.append((prod, city, missing_days))

# --- Summary across all combos ---
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Total combinations checked : {len(combos)}")
print(f"  Combos with missing dates   : {len(combos_with_gaps)}")
if combos_with_gaps:
    print(f"\n  ⚠️ The following combos have gaps:")
    for prod, city, missing in combos_with_gaps[:20]:
        print(f"    {prod} / {city}: {missing} missing days")
    if len(combos_with_gaps) > 20:
        print(f"    ... and {len(combos_with_gaps) - 20} more")
else:
    print("  ✅ All combinations have continuous dates.")

In [0]:
# ============================================================
# CELL 6: Verify lag_7 and rolling_7 Semantics
# ============================================================
# This is the most important data-quality check for a forecasting
# project. We determine whether:
#   • demand_lag_1  = total_demand 1 CALENDAR day ago  (correct)
#     or total_demand 1 ROW ago  (wrong if dates have gaps)
#   • demand_lag_7  = total_demand 7 CALENDAR days ago (correct)
#     or total_demand 7 ROWS ago (wrong if dates have gaps)
#   • rolling_avg_7 = average of total_demand over the previous
#     7 calendar days. Does it include or exclude the current day?
#
# APPROACH:
#   1. Pick one product-city combination
#   2. Collect all rows into a Python dict keyed by date
#   3. For several dates D, look up the actual total_demand at
#      D-1 and D-7 and compare with the stored lag values
#
# WHAT TO VERIFY after running:
#   • demand_lag_1 should MATCH total_demand from 1 calendar day ago
#   • demand_lag_7 should MATCH total_demand from 7 calendar days ago
#   • If they DON'T match, the lag features were computed with a
#     row-based shift (not calendar-based) — this is a problem if
#     dates have gaps. You'll need to recompute them correctly.
#   • rolling_avg_7 should be the mean of the previous 7 calendar
#     days' total_demand (check if it includes the current day).
# ------------------------------------------------------------

import datetime

print("=" * 60)
print("LAG & ROLLING SEMANTICS VERIFICATION")
print("=" * 60)

# --- Pick the first product-city combination ---
sample_combo = combos[0]
sample_prod = sample_combo["product_name"]
sample_city = sample_combo["destination_city"]

print(f"\nTesting combination: {sample_prod} / {sample_city}")

# --- Collect all rows for this combo, ordered by date ---
sample_rows = (feat_df
    .filter(
        (F.col("product_name") == sample_prod) &
        (F.col("destination_city") == sample_city)
    )
    .orderBy("transaction_date")
    .select(
        "transaction_date", "total_demand",
        "demand_lag_1", "demand_lag_7",
        "rolling_avg_7", "rolling_std_7",
    )
    .collect())

print(f"Total rows for this combo: {len(sample_rows)}")

# --- Build a lookup dict: date → row ---
data_by_date = {row["transaction_date"]: row for row in sample_rows}

# --- Verify lag_1 and lag_7 for several dates ---
# Skip the first 8 rows (they won't have enough history for lag_7)
print("\n--- Checking demand_lag_1 (should = total_demand 1 calendar day ago) ---")
print("--- Checking demand_lag_7 (should = total_demand 7 calendar days ago) ---\n")

lag1_matches = 0
lag1_checked = 0
lag7_matches = 0
lag7_checked = 0

for row in sample_rows[8:20]:  # Check up to 12 rows
    d = row["transaction_date"]
    d_minus_1 = d - datetime.timedelta(days=1)
    d_minus_7 = d - datetime.timedelta(days=7)

    lag1_val = row["demand_lag_1"]
    lag7_val = row["demand_lag_7"]

    # Look up actual demand on those past dates
    actual_d1 = data_by_date.get(d_minus_1)
    actual_d7 = data_by_date.get(d_minus_7)

    actual_demand_d1 = actual_d1["total_demand"] if actual_d1 else None
    actual_demand_d7 = actual_d7["total_demand"] if actual_d7 else None

    # Check matches
    lag1_ok = (lag1_val is not None and actual_demand_d1 is not None and lag1_val == actual_demand_d1)
    lag7_ok = (lag7_val is not None and actual_demand_d7 is not None and lag7_val == actual_demand_d7)

    if actual_demand_d1 is not None:
        lag1_checked += 1
        if lag1_ok:
            lag1_matches += 1
    if actual_demand_d7 is not None:
        lag7_checked += 1
        if lag7_ok:
            lag7_matches += 1

    print(f"Date: {d}")
    print(f"  demand_lag_1 = {lag1_val}  |  actual demand {d_minus_1} = {actual_demand_d1}  |  {'✅ MATCH' if lag1_ok else '❌ MISMATCH' if actual_demand_d1 is not None else '⚠️ date missing'}")
    print(f"  demand_lag_7 = {lag7_val}  |  actual demand {d_minus_7} = {actual_demand_d7}  |  {'✅ MATCH' if lag7_ok else '❌ MISMATCH' if actual_demand_d7 is not None else '⚠️ date missing'}")
    print()

# --- Summary ---
print("=" * 60)
print("LAG SEMANTICS SUMMARY")
print("=" * 60)
print(f"  demand_lag_1: {lag1_matches}/{lag1_checked} matched calendar-day lookup")
print(f"  demand_lag_7: {lag7_matches}/{lag7_checked} matched calendar-day lookup")

if lag1_matches == lag1_checked and lag7_matches == lag7_checked:
    print("  ✅ Lag features are CALENDAR-DAY based (correct for forecasting).")
else:
    print("  ⚠️ Some lags did NOT match calendar-day lookups.")
    print("  The lag features may have been computed with a ROW-BASED shift.")
    print("  If dates have gaps, these lag values are INCORRECT and must be recomputed.")

# --- Quick check: does rolling_avg_7 include the current day? ---
print("\n--- Checking rolling_avg_7 (current day included or excluded?) ---")
# Pick a date with enough history
test_row = sample_rows[10]
test_date = test_row["transaction_date"]
rolling_val = test_row["rolling_avg_7"]

# Compute average of previous 7 days EXCLUDING current day
prev_7_excl = []
prev_7_incl = []
for offset in range(1, 8):  # D-1 to D-7
    check_date = test_date - datetime.timedelta(days=offset)
    if check_date in data_by_date:
        prev_7_excl.append(data_by_date[check_date]["total_demand"])
for offset in range(0, 7):  # D to D-6
    check_date = test_date - datetime.timedelta(days=offset)
    if check_date in data_by_date:
        prev_7_incl.append(data_by_date[check_date]["total_demand"])

avg_excl = sum(prev_7_excl) / len(prev_7_excl) if prev_7_excl else None
avg_incl = sum(prev_7_incl) / len(prev_7_incl) if prev_7_incl else None

print(f"  Date: {test_date}")
print(f"  rolling_avg_7 stored value         : {rolling_val}")
print(f"  Avg of D-1..D-7 (excl current day) : {avg_excl}")
print(f"  Avg of D..D-6  (incl current day)   : {avg_incl}")

if rolling_val is not None and avg_excl is not None and abs(rolling_val - avg_excl) < 0.01:
    print("  → rolling_avg_7 EXCLUDES the current day (previous 7 days).")
elif rolling_val is not None and avg_incl is not None and abs(rolling_val - avg_incl) < 0.01:
    print("  → rolling_avg_7 INCLUDES the current day.")
    print("  ⚠️ If it includes the current day's demand, this is fine for")
    print("  7-day-ahead forecasting (current day is known at prediction time).")
else:
    print("  → Could not determine — may use a different window definition.")

In [0]:
# ============================================================
# CELL 7: Potential Leakage Assessment & Summary
# ============================================================
# Before moving to Step 2 (target creation), we classify every
# column and think about what information would be available at
# prediction time vs. what would constitute leakage.
#
# KEY LEAKAGE CONCEPT FOR THIS PROJECT:
#   • We are forecasting demand 7 calendar days AHEAD.
#   • At prediction time (today = origin_date), we KNOW:
#     - today's total_demand, avg_unit_price, total_inventory
#     - today's lags and rolling stats (they use past data)
#   • We do NOT know the demand 7 days from now — that is the TARGET.
#   • So total_demand on the origin date is a VALID feature,
#     but total_demand on the target date is the TARGET (must
#     never appear in features).
#
# WHAT TO VERIFY after running:
#   • All feature columns use only past/current information.
#   • No column is a proxy for the future target.
#   • You have a clear written summary of all data characteristics
#     — enough to explain it in an interview.
# ------------------------------------------------------------

print("=" * 60)
print("LEAKAGE ASSESSMENT & DATA SUMMARY")
print("=" * 60)

# --- Classify every column in demand_features ---
column_classification = {
    # Column name: (role, available at prediction time?, notes)
    "transaction_date":    ("Key (origin date)",     "Yes", "The date we make the prediction FROM"),
    "product_name":        ("Key (categorical)",    "Yes", "Identifies the product — used for grouping"),
    "destination_city":   ("Key (categorical)",    "Yes", "Identifies the city — used for grouping"),
    "total_demand":       ("Feature (numeric)",    "Yes", "Today's demand — known at prediction time ✓"),
    "avg_unit_price":      ("Feature (numeric)",    "Yes", "Today's avg price — known at prediction time ✓"),
    "total_inventory":    ("Feature (numeric)",    "Yes", "Today's inventory — known at prediction time ✓"),
    "day_of_week":         ("Feature (numeric)",    "Yes", "Derived from date — always available ✓"),
    "month":              ("Feature (numeric)",    "Yes", "Derived from date — always available ✓"),
    "demand_lag_1":        ("Feature (numeric)",    "Yes", "Past demand (1 day ago) — safe ✓"),
    "demand_lag_7":        ("Feature (numeric)",    "Yes", "Past demand (7 days ago) — safe ✓"),
    "rolling_avg_7":      ("Feature (numeric)",    "Yes", "7-day rolling avg of past demand — safe ✓"),
    "rolling_std_7":      ("Feature (numeric)",    "Yes", "7-day rolling std of past demand — safe ✓"),
}

print("\n--- Column Classification (demand_features) ---")
print(f"  {'Column':<22} {'Role':<22} {'Available?':<12} Notes")
print(f"  {'-'*22} {'-'*22} {'-'*12} {'-'*40}")
for col_name, (role, available, notes) in column_classification.items():
    print(f"  {col_name:<22} {role:<22} {available:<12} {notes}")

# --- Leakage checklist ---
print("\n--- Leakage Checklist ---")
print("  ✓ total_demand (origin date) — known today, valid feature")
print("  ✓ All lag features — use past data only")
print("  ✓ Rolling features — use past data only")
print("  ✓ day_of_week, month — derived from date, always available")
print("  ⚠️  In Step 2, target_demand (demand 7 days ahead) will be")
print("     created by joining daily_demand_agg on target_date.")
print("     It must NEVER appear in the feature set.")
print("  ⚠️  Ensure no future-dated rows leak into training features.")

# --- Final summary dictionary ---
print("\n" + "=" * 60)
print("FINAL DATA SUMMARY (for interview reference)")
print("=" * 60)

summary = {
    "Table":                    "oil_gas_demo.gold.demand_features",
    "Total rows":               f"{feat_summary['row_count']:,}",
    "Date range":               f"{feat_summary['min_date']} → {feat_summary['max_date']}",
    "Product-city combinations": f"{feat_summary['n_combos']}",
    "Distinct products":       f"{feat_summary['n_products']}",
    "Distinct cities":         f"{feat_summary['n_cities']}",
    "Duplicate keys":          "0" if len(dup_rows) == 0 else f"{len(dup_rows)} (⚠️ resolve!)",
    "Combos with date gaps":    f"{len(combos_with_gaps)}",
    "Lag semantics":            "Calendar-day based ✅" if (lag1_matches == lag1_checked and lag7_matches == lag7_checked) else "⚠️ Check mismatch",
    "Leakage suspects":        "None identified",
    "Safe for Pandas (< 500K)?": "Yes" if feat_summary['row_count'] < 500000 else "⚠️ May need sampling",
}

for key, value in summary.items():
    print(f"  {key:<30} : {value}")

print("""
📌 INTERVIEW EXPLANATION:
"Before training, I inspected both Gold tables to understand the data.
I verified the runtime environment, confirmed scikit-learn and MLflow
were available, and checked the Feature Engineering API. I validated
schemas, row counts, and date ranges. I confirmed the composite key
(date, product, city) had no duplicates. I checked date continuity
per product-city combination to ensure lags and rolling averages
were computed on calendar days, not row offsets. Finally, I assessed
potential leakage — all lag and rolling features use past data and are
safe, while total_demand on the origin date is a valid feature but on
the target date (7 days ahead) is the prediction target that must
never appear in features."
""")